# ESC-50 — MOO + TSK v23: Comparativo de Algoritmos Multiobjetivo com Agregador Fuzzy

## Objetivo
Comparar **3 algoritmos de otimização multiobjetivo** aplicados ao mesmo problema de otimização de hiperparâmetros, agora utilizando o **ESC-50**, cada um executado **com** e **sem** o Agregador Fuzzy (TSK). A evidência principal é a participação de cada variante na fronteira de Pareto unificada final.

## Algoritmos comparados

| # | Algoritmo | Tipo | Estratégia de seleção | Referência |
|---|---|---|---|---|
| 1 | **NSGA-II** | Dominância de Pareto + crowding | Ranking + distância de aglomeração | Deb et al. 2002 |
| 2 | **NSGA-III** | Dominância de Pareto + pontos de referência | Ranking + projeção em hiperplano | Deb & Jain 2014 |
| 3 | **MOEA/D** | Decomposição por pesos | Otimização escalarizada por vizinhança | Zhang & Li 2007 |

## Estrutura experimental
- Cada algoritmo roda **2 vezes**: com Fuzzy (4 objetivos: `[1-acc, custo, 1-auroc, 1-fis_quality]`) e sem Fuzzy (3 objetivos: `[1-acc, custo, 1-auroc]`).
- Total: **6 rodadas independentes**.
- As arquiteturas convolucionais, o cromossomo, os operadores evolutivos e os hiperparâmetros da busca são mantidos em relação ao experimento-modelo.
- O ESC-50 é dividido por seus **folds oficiais**, sem divisão aleatória: folds 1–3 para treino, fold 4 para validação/otimização e fold 5 como teste final reservado.
- Cache global compartilhado: uma configuração já avaliada deterministicamente é reutilizada entre algoritmos e variantes.
- A fronteira de Pareto unificada é calculada no espaço comum de 3 objetivos para as 6 rodadas.
- As métricas e visualizações finais seguem o notebook `visualizations.ipynb`.

## Observação metodológica
Os perfis e regras do sistema TSK foram mantidos para isolar o efeito da mudança de dataset. Caso esses limiares tenham sido calibrados especificamente no GTZAN, recomenda-se apresentar também uma análise de sensibilidade ou uma calibração independente no ESC-50.

In [ ]:
# [0] Instalação
!pip install torch torchvision pymoo scikit-learn numpy pandas matplotlib scipy librosa soundfile

In [2]:
# [1] Imports
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os, random, time, gc, json, hashlib, warnings
from pathlib import Path
from itertools import product as iproduct
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, classification_report
)
# ── Algoritmos MOO ──
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2  import NSGA2
from pymoo.algorithms.moo.nsga3  import NSGA3
from pymoo.algorithms.moo.moead  import MOEAD
from pymoo.algorithms.moo.sms    import SMSEMOA
from pymoo.algorithms.moo.rnsga2 import RNSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm   import PM
from pymoo.operators.sampling.rnd  import FloatRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.core.callback import Callback
from pymoo.indicators.hv import HV
from pymoo.util.ref_dirs import get_reference_directions
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Device: cuda
  GPU : NVIDIA GeForce RTX 3050 Laptop GPU
  VRAM: 4.3 GB


In [ ]:
# [2] Configuração
# Caminhos portáveis: use a estrutura padrão do projeto ou variáveis de ambiente.
def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'README.md').is_file() and (candidate / 'esc50').is_dir():
            return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada. Inicie o Jupyter dentro do repositório.')

PROJECT_ROOT = find_project_root()
DATASETS_ROOT = Path(
    os.environ.get('PYMMOO_DATASETS_DIR', PROJECT_ROOT / 'datasets')
).expanduser().resolve()
ESC50_ROOT = Path(
    os.environ.get('ESC50_ROOT', DATASETS_ROOT / 'ESC-50-master')
).expanduser().resolve()
META_DIR   = ESC50_ROOT / 'meta'
META_CSV   = META_DIR / 'esc50.csv'
AUDIO_ROOT = ESC50_ROOT / 'audio'

DATASET_NAME = 'esc50_v23'
AUDIO_SR     = 22050   # reamostragem para manter o custo controlado
CLIP_SECONDS = 5.0     # duração oficial dos clipes do ESC-50

# Divisão fixa baseada nos folds oficiais do ESC-50.
# O fold de teste não participa da otimização.
TRAIN_FOLDS = (1, 2, 3)
VAL_FOLDS   = (4,)
TEST_FOLDS  = (5,)

# ── Hiperparâmetros do loop evolutivo — iguais para todos os algoritmos ──
POP     = 30
GEN     = 15
SBX_ETA = 10
PM_ETA  = 15

BASE_DIR  = PROJECT_ROOT / 'results' / DATASET_NAME
CKPT_DIR  = BASE_DIR / 'checkpoints'
PLOTS_DIR = BASE_DIR / 'plots'
LOGS_DIR  = BASE_DIR / 'logs'
CSV_DIR   = BASE_DIR / 'csv'
for d in [CKPT_DIR, PLOTS_DIR, LOGS_DIR, CSV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ALGORITHMS = [
    ('nsga2', 'NSGA-II'),
    ('nsga3', 'NSGA-III'),
    ('moead', 'MOEA/D'),
]
VARIANTS = ['fuzzy', 'puro']

print(f'Metadados: {META_CSV}')
print(f'Áudios: {AUDIO_ROOT}')
print(f'Folds: treino={TRAIN_FOLDS}, validação={VAL_FOLDS}, teste={TEST_FOLDS}')
print(f'Algoritmos: {[a[1] for a in ALGORITHMS]}')
print(f'Variantes: {VARIANTS}')
print(f'Total de rodadas: {len(ALGORITHMS) * len(VARIANTS)}')
print(f'pop={POP}  gen={GEN}')

In [4]:
# [3] Dataset ESC-50
TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

if not META_CSV.exists():
    raise FileNotFoundError(
        f'CSV de metadados não encontrado em: {META_CSV}\n'
        'Ajuste somente a variável META_DIR na célula de configuração.'
    )
if not AUDIO_ROOT.exists():
    raise FileNotFoundError(
        f'Pasta de áudios não encontrada em: {AUDIO_ROOT}\n'
        'A estrutura esperada é ESC-50-master/audio e ESC-50-master/meta.'
    )

ESC50_META = pd.read_csv(META_CSV)
REQUIRED_COLUMNS = {'filename', 'fold', 'target', 'category'}
missing_columns = REQUIRED_COLUMNS.difference(ESC50_META.columns)
if missing_columns:
    raise ValueError(f'Colunas ausentes no esc50.csv: {sorted(missing_columns)}')

ESC50_META = ESC50_META.sort_values(['fold', 'filename']).reset_index(drop=True)
TARGET_TO_CATEGORY = (
    ESC50_META[['target', 'category']]
    .drop_duplicates()
    .sort_values('target')
    .set_index('target')['category']
    .to_dict()
)
CLASSES = [TARGET_TO_CATEGORY[i] for i in sorted(TARGET_TO_CATEGORY)]
NUM_CLASSES = len(CLASSES)

if sorted(ESC50_META['fold'].unique().tolist()) != [1, 2, 3, 4, 5]:
    raise ValueError('Os metadados não contêm os cinco folds oficiais do ESC-50.')
if NUM_CLASSES != 50:
    raise ValueError(f'Esperadas 50 classes, encontradas {NUM_CLASSES}.')

missing_audio = [
    str(AUDIO_ROOT / filename)
    for filename in ESC50_META['filename']
    if not (AUDIO_ROOT / filename).exists()
]
if missing_audio:
    raise FileNotFoundError(
        f'{len(missing_audio)} arquivos de áudio não foram encontrados. '
        f'Primeiro exemplo: {missing_audio[0]}'
    )

class ESC50WavDataset(Dataset):
    def __init__(
        self,
        metadata,
        audio_root,
        folds,
        n_mels=128,
        n_fft=2048,
        hop_length=512,
        fmax=8000,
        transform=None,
        sample_rate=AUDIO_SR,
        duration=CLIP_SECONDS,
    ):
        try:
            import librosa
            self._lib = librosa
        except ImportError as exc:
            raise ImportError('Instale librosa e soundfile.') from exc

        self.audio_root = Path(audio_root)
        self.transform = transform
        self.n_mels = int(n_mels)
        self.n_fft = int(n_fft)
        self.hop_length = int(hop_length)
        self.fmax = float(fmax)
        self.sample_rate = int(sample_rate)
        self.duration = float(duration)
        self.folds = tuple(int(fold) for fold in folds)

        subset = metadata[metadata['fold'].isin(self.folds)].copy()
        if subset.empty:
            raise ValueError(f'Nenhum exemplo encontrado para os folds {self.folds}.')

        self.samples = [
            (self.audio_root / row.filename, int(row.target))
            for row in subset.itertuples(index=False)
        ]
        self.classes = CLASSES

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        y, _ = self._lib.load(
            path,
            sr=self.sample_rate,
            mono=True,
            duration=self.duration,
        )

        expected_samples = int(self.sample_rate * self.duration)
        if len(y) < expected_samples:
            y = np.pad(y, (0, expected_samples - len(y)))
        elif len(y) > expected_samples:
            y = y[:expected_samples]

        spectrogram = self._lib.feature.melspectrogram(
            y=y,
            sr=self.sample_rate,
            n_mels=self.n_mels,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            fmax=min(self.fmax, self.sample_rate / 2),
        )
        spectrogram_db = self._lib.power_to_db(spectrogram, ref=np.max)
        minimum, maximum = spectrogram_db.min(), spectrogram_db.max()
        spectrogram_norm = (spectrogram_db - minimum) / (maximum - minimum + 1e-8)
        image = Image.fromarray(
            (np.stack([spectrogram_norm] * 3, axis=2) * 255).astype(np.uint8)
        )

        if self.transform:
            image = self.transform(image)
        return image, label

print(f'ESC-50: {len(ESC50_META)} arquivos, {NUM_CLASSES} classes')
print('Arquivos por fold:')
print(ESC50_META.groupby('fold').size().to_string())
print(f'Treino: {ESC50_META[ESC50_META.fold.isin(TRAIN_FOLDS)].shape[0]}')
print(f'Validação: {ESC50_META[ESC50_META.fold.isin(VAL_FOLDS)].shape[0]}')
print(f'Teste: {ESC50_META[ESC50_META.fold.isin(TEST_FOLDS)].shape[0]}')

BACKBONE_CFGS = {
    0: ('ResNet18', models.resnet18, 'IMAGENET1K_V1', 512),
    1: ('EfficientNet-B0', models.efficientnet_b0, 'IMAGENET1K_V1', 1280),
    2: ('MobileNetV2', models.mobilenet_v2, 'IMAGENET1K_V1', 1280),
}
LATENCY_REF = {'ResNet18': 10, 'EfficientNet-B0': 12, 'MobileNetV2': 8}
LAT_MAX = max(LATENCY_REF.values())
COST_MAX = 80 * 12

ESC-50: 2000 arquivos, 50 classes
Arquivos por fold:
fold
1    400
2    400
3    400
4    400
5    400
Treino: 1200
Validação: 400
Teste: 400


In [5]:
# [4] Cromossomo, decode e utilitários
N_VAR=11
xl=np.array([0.0,0,20,0.0,0,0,0.1, 0,0,0.1,0])
xu=np.array([1.0,2,80,1.0,2,2,0.5, 3,2,0.5,2])
BATCH_SIZES=[32,64,128]
N_MELS_OPTS=[32,64,128,256]
N_FFT_OPTS=[512,1024,2048]
FMAX_OPTS=[4000,8000,11025]

def decode(x):
    lr=10**(np.log10(1e-4)+x[0]*(np.log10(1e-1)-np.log10(1e-4)))
    wd=10**(np.log10(1e-5)+x[3]*(np.log10(1e-2)-np.log10(1e-5)))
    n_fft=N_FFT_OPTS[int(np.clip(round(x[8]),0,2))]
    hop=max(64,int(n_fft*float(np.clip(x[9],0.1,0.5))))
    return {
        'lr':float(lr),'optimizer':int(np.clip(round(x[1]),0,2)),
        'epochs':int(np.clip(round(x[2]),20,80)),'wd':float(wd),
        'backbone_idx':int(np.clip(round(x[4]),0,2)),
        'batch_size':BATCH_SIZES[int(np.clip(round(x[5]),0,2))],
        'dropout':float(np.clip(x[6],0.1,0.5)),
        'n_mels':N_MELS_OPTS[int(np.clip(round(x[7]),0,3))],
        'n_fft':n_fft,'hop_length':hop,
        'fmax':FMAX_OPTS[int(np.clip(round(x[10]),0,2))],
    }

def xkey(x): return hashlib.md5(np.array(x,dtype=np.float64).tobytes()).hexdigest()

HV_REF3=np.array([1.1,1.1,1.1])
HV_REF4=np.array([1.1,1.1,1.1,1.1])

def pareto_filter(F):
    n=len(F); dom=np.zeros(n,dtype=bool)
    for i in range(n):
        for j in range(n):
            if i==j: continue
            if np.all(F[j]<=F[i]) and np.any(F[j]<F[i]): dom[i]=True; break
    return ~dom

def calc_hv(F_arc, ref=None):
    r=ref if ref is not None else HV_REF3
    return float(HV(ref_point=r).do(F_arc))

def compute_cost(m):
    return float(np.clip(m['params']['epochs']*LATENCY_REF[m['backbone']]/COST_MAX,0,1))

N_MELS_MAX=256; N_FFT_MAX=2048
def spec_cost_norm(p):
    return float((p['n_mels']/N_MELS_MAX)*(p['n_fft']/N_FFT_MAX))

EP_MIN,EP_MAX=20,80
print(f'Cromossomo: {N_VAR} genes | objetivos brutos: [1-acc, custo, 1-auroc]')

Cromossomo: 11 genes | objetivos brutos: [1-acc, custo, 1-auroc]


In [6]:
# [5] LinearHead + fast_eval (cache global compartilhado entre TODAS as rodadas)
class LinearHead(nn.Module):
    def __init__(self,feat_dim,num_classes,dropout=0.4):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(feat_dim,512),nn.BatchNorm1d(512),nn.ReLU(),nn.Dropout(dropout),
            nn.Linear(512,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(dropout*0.75),
            nn.Linear(256,num_classes)
        )
    def forward(self,x): return self.net(x)

# Cache GLOBAL — compartilhado entre TODOS os algoritmos e variantes.
# Correto metodologicamente: a avaliação é determinística; reutilizá-la
# não favorece nenhum algoritmo específico.
EVAL_CACHE = {}

def fast_eval(x_vec):
    key = xkey(x_vec)
    if key in EVAL_CACHE:
        m = EVAL_CACHE[key]
        print(f"  CACHE {m['backbone']:18s}  acc={m['accuracy']:.4f}", flush=True)
        return m

    params = decode(x_vec)
    bidx = params['backbone_idx']
    bname = BACKBONE_CFGS[bidx][0]
    feat_dim = BACKBONE_CFGS[bidx][3]

    dataset_kwargs = dict(
        metadata=ESC50_META,
        audio_root=AUDIO_ROOT,
        n_mels=params['n_mels'],
        n_fft=params['n_fft'],
        hop_length=params['hop_length'],
        fmax=params['fmax'],
        transform=TRANSFORM,
    )
    tr_ds = ESC50WavDataset(folds=TRAIN_FOLDS, **dataset_kwargs)
    va_ds = ESC50WavDataset(folds=VAL_FOLDS, **dataset_kwargs)
    te_ds = ESC50WavDataset(folds=TEST_FOLDS, **dataset_kwargs)

    print(
        f"  TRAIN {bname:18s} lr={params['lr']:.2e} ep={params['epochs']} "
        f"[{params['n_mels']}mel/{params['n_fft']}fft] "
        f"split={len(tr_ds)}/{len(va_ds)}/{len(te_ds)}",
        flush=True,
    )

    def _extract_features(dataset):
        base = BACKBONE_CFGS[bidx][1](weights=BACKBONE_CFGS[bidx][2]).to(device)
        if hasattr(base, 'fc'):
            base.fc = nn.Identity()
        elif hasattr(base, 'classifier'):
            base.classifier = nn.Identity()
        base.eval()

        loader = DataLoader(dataset, 64, shuffle=False, num_workers=0)
        features, labels = [], []
        with torch.no_grad():
            for images, targets in loader:
                features.append(base(images.to(device)).cpu())
                labels.append(targets if torch.is_tensor(targets) else torch.tensor(targets))

        del base
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return torch.cat(features), torch.cat(labels)

    f_tr, l_tr = _extract_features(tr_ds)
    f_va, l_va = _extract_features(va_ds)
    f_te, l_te = _extract_features(te_ds)

    seed = int(key[:8], 16) % (2**31)
    torch.manual_seed(seed)
    np.random.seed(seed % (2**32))

    batch_size = params['batch_size']
    train_generator = torch.Generator().manual_seed(seed)
    tr_ld = DataLoader(
        TensorDataset(f_tr, l_tr),
        batch_size,
        shuffle=True,
        generator=train_generator,
    )
    va_ld = DataLoader(TensorDataset(f_va, l_va), batch_size, shuffle=False)

    model = LinearHead(feat_dim, NUM_CLASSES, params['dropout']).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer_kwargs = {'lr': params['lr'], 'weight_decay': params['wd']}
    if params['optimizer'] == 2:
        optimizer_kwargs.update({'momentum': 0.9, 'nesterov': True})
    optimizer_class = {0: optim.Adam, 1: optim.AdamW, 2: optim.SGD}[params['optimizer']]
    optimizer = optimizer_class(model.parameters(), **optimizer_kwargs)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=params['epochs'],
        eta_min=params['lr'] * 0.01,
    )

    best_loss, patience = float('inf'), 0
    for _ in range(params['epochs']):
        model.train()
        for features, targets in tr_ld:
            features, targets = features.to(device), targets.to(device)
            optimizer.zero_grad()
            criterion(model(features), targets).backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            validation_loss = sum(
                criterion(model(features.to(device)), targets.to(device)).item()
                for features, targets in va_ld
            ) / len(va_ld)

        if validation_loss < best_loss:
            best_loss = validation_loss
            patience = 0
        else:
            patience += 1
        if patience >= 6:
            break

    model.eval()
    predictions, labels, probabilities = [], [], []
    with torch.no_grad():
        for features, targets in va_ld:
            output = model(features.to(device))
            predictions.extend(torch.argmax(output, 1).cpu().numpy())
            labels.extend(targets.numpy())
            probabilities.extend(torch.softmax(output, 1).cpu().numpy())

    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)

    accuracy = float(accuracy_score(labels, predictions))
    f1_macro = float(f1_score(labels, predictions, average='macro', zero_division=0))
    precision = float(precision_score(labels, predictions, average='macro', zero_division=0))
    recall = float(recall_score(labels, predictions, average='macro', zero_division=0))
    try:
        auroc = float(roc_auc_score(labels, probabilities, multi_class='ovr', average='macro'))
    except ValueError:
        auroc = 0.5

    result = {
        'accuracy': accuracy,
        'f1': f1_macro,
        'precision': precision,
        'recall': recall,
        # Mantido por compatibilidade com o modelo original: aqui representa taxa de erro.
        'fpr': 1 - accuracy,
        'auroc': auroc,
        'loss': best_loss,
        'backbone': bname,
        'backbone_idx': bidx,
        'params': params,
        'model_state': {name: value.cpu() for name, value in model.state_dict().items()},
        'dropout': params['dropout'],
        'feat_dim': feat_dim,
        'f_te': f_te,
        'l_te': l_te,
        'split': {
            'train_folds': TRAIN_FOLDS,
            'val_folds': VAL_FOLDS,
            'test_folds': TEST_FOLDS,
        },
    }

    del model, optimizer, scheduler
    gc.collect()
    EVAL_CACHE[key] = result
    print(
        f'        acc={accuracy:.4f} f1={f1_macro:.4f} auroc={auroc:.4f}',
        flush=True,
    )
    return result

def eval_on_test(m):
    te_ld=DataLoader(TensorDataset(m['f_te'],m['l_te']),256,shuffle=False)
    model=LinearHead(m['feat_dim'],NUM_CLASSES,m.get('dropout',0.4)).to(device)
    model.load_state_dict({k:v.to(device) for k,v in m['model_state'].items()})
    model.eval(); preds,labs,probs=[],[],[]
    with torch.no_grad():
        for X,y in te_ld:
            out=model(X.to(device))
            preds.extend(torch.argmax(out,1).cpu().numpy())
            labs.extend(y.numpy()); probs.extend(torch.softmax(out,1).cpu().numpy())
    preds=np.array(preds); labs=np.array(labs); probs=np.array(probs)
    acc=float(accuracy_score(labs,preds))
    f1m=float(f1_score(labs,preds,average='macro',zero_division=0))
    try:    auroc=float(roc_auc_score(labs,probs,multi_class='ovr',average='macro'))
    except: auroc=0.5
    del model; gc.collect()
    return {'accuracy':acc,'f1':f1m,'auroc':auroc,'preds':preds,'labels':labs,'probs':probs}

print('fast_eval OK — folds oficiais e cache global compartilhado entre as 6 rodadas')

fast_eval OK — folds oficiais e cache global compartilhado entre as 6 rodadas


In [7]:
# [6] TSK Adaptativo — FIS-Eval, FIS-OOD, FIS-Perf, FIS-Sel (idêntico ao v21)
STATE_NAMES=['very_low','low','medium','high','very_high']
PROFILE={
    'acc':    [(0.00,0.65),(0.65,0.72),(0.72,0.79),(0.79,0.85),(0.85,1.00)],
    'f1':     [(0.00,0.65),(0.65,0.72),(0.72,0.79),(0.79,0.85),(0.85,1.00)],
    'auroc':  [(0.80,0.87),(0.87,0.91),(0.91,0.94),(0.94,0.97),(0.97,1.00)],
    'fpr_inv':[(0.00,0.645),(0.645,0.715),(0.715,0.785),(0.785,0.845),(0.845,1.000)],
}
def build_mfs(profile,tightness=2.5):
    c=[(lo+hi)/2 for lo,hi in profile]
    s=[max((hi-lo)/tightness,0.006) for lo,hi in profile]
    return c,s
def membership(x,cs,ss):
    return np.array([np.exp(-((x-c)**2)/(2*sg**2)) for c,sg in zip(cs,ss)])
C_ACC,S_ACC=build_mfs(PROFILE['acc']); C_F1,S_F1=build_mfs(PROFILE['f1'])
C_AU,S_AU=build_mfs(PROFILE['auroc']); C_FP,S_FP=build_mfs(PROFILE['fpr_inv'])
C_STD,S_STD=build_mfs([(0.0,0.2),(0.2,0.4),(0.4,0.6),(0.6,0.8),(0.8,1.0)],tightness=2.0)
TSK_EVAL={'very_high':(+0.20,0.55,0.45),'high':(+0.08,0.55,0.45),'medium':(0.00,0.50,0.50),'low':(-0.12,0.50,0.50),'very_low':(-0.30,0.45,0.55)}
TSK_OOD ={'very_high':(+0.12,0.50,0.50),'high':(+0.05,0.50,0.50),'medium':(0.00,0.50,0.50),'low':(-0.08,0.50,0.50),'very_low':(-0.18,0.50,0.50)}
TSK_PERF={'very_high':(+0.10,0.30,0.30,0.20,0.20),'high':(+0.04,0.30,0.30,0.20,0.20),'medium':(0.00,0.30,0.30,0.20,0.20),'low':(-0.06,0.30,0.30,0.20,0.20),'very_low':(-0.16,0.30,0.30,0.20,0.20)}
TSK_SEL ={'very_high':(+0.08,0.50,0.30,0.20),'high':(+0.03,0.50,0.30,0.20),'medium':(0.00,0.50,0.30,0.20),'low':(-0.06,0.50,0.30,0.20),'very_low':(-0.18,0.50,0.30,0.20)}
def tsk_2in(x1,x2,c1,s1,c2,s2,coeff):
    mu1=membership(x1,c1,s1); mu2=membership(x2,c2,s2); tw,twz=0.0,0.0
    for i,si in enumerate(STATE_NAMES):
        for j,sj in enumerate(STATE_NAMES):
            w=float(mu1[i]*mu2[j])
            if w<1e-12: continue
            ci=coeff[si]; cj=coeff[sj]; a0=(ci[0]+cj[0])/2; a1=(ci[1]+cj[1])/2; a2=(ci[2]+cj[2])/2
            twz+=w*(a0+a1*x1+a2*x2); tw+=w
    return float(np.clip(twz/tw,0,1)) if tw>0 else float((x1+x2)/2)
def tsk_3in(x1,x2,x3,c1,s1,c2,s2,c3,s3,coeff):
    mu1=membership(x1,c1,s1); mu2=membership(x2,c2,s2); mu3=membership(x3,c3,s3); tw,twz=0.0,0.0
    for i,si in enumerate(STATE_NAMES):
        for j,sj in enumerate(STATE_NAMES):
            for k in range(len(STATE_NAMES)):
                w=float(mu1[i]*mu2[j]*mu3[k])
                if w<1e-12: continue
                ci=coeff[si]; cj=coeff[sj]; ck=coeff[STATE_NAMES[k]]
                a0=(ci[0]+cj[0]+ck[0])/3; a1=(ci[1]+cj[1]+ck[1])/3; a2=(ci[2]+cj[2]+ck[2])/3; a3=(ci[3]+cj[3]+ck[3])/3
                twz+=w*(a0+a1*x1+a2*x2+a3*x3); tw+=w
    return float(np.clip(twz/tw,0,1)) if tw>0 else float((x1+x2+x3)/3)
def tsk_4in(x1,x2,x3,x4,c1,s1,c2,s2,c3,s3,c4,s4,coeff):
    mu1=membership(x1,c1,s1); mu2=membership(x2,c2,s2); mu3=membership(x3,c3,s3); mu4=membership(x4,c4,s4); tw,twz=0.0,0.0
    for i,si in enumerate(STATE_NAMES):
      for j,sj in enumerate(STATE_NAMES):
        for k,sk in enumerate(STATE_NAMES):
          for l_idx,sl in enumerate(STATE_NAMES):
            w=float(mu1[i]*mu2[j]*mu3[k]*mu4[l_idx])
            if w<1e-12: continue
            ci=coeff[si]; cj=coeff[sj]; ck=coeff[sk]; cl_=coeff[sl]
            a0=(ci[0]+cj[0]+ck[0]+cl_[0])/4; a1=(ci[1]+cj[1]+ck[1]+cl_[1])/4
            a2=(ci[2]+cj[2]+ck[2]+cl_[2])/4; a3=(ci[3]+cj[3]+ck[3]+cl_[3])/4; a4=(ci[4]+cj[4]+ck[4]+cl_[4])/4
            twz+=w*(a0+a1*x1+a2*x2+a3*x3+a4*x4); tw+=w
    return float(np.clip(twz/tw,0,1)) if tw>0 else float((x1+x2+x3+x4)/4)
def perf_inputs(m):
    ep=m['params']['epochs']; drop=m['params']['dropout']
    lat_b=LATENCY_REF[m['backbone']]/LAT_MAX
    lat_s=float(np.clip(1-lat_b*(ep/EP_MAX),0,1))
    comp_s=float(np.clip(1-(ep-EP_MIN)/(EP_MAX-EP_MIN),0,1))
    reg_s=float(np.clip(drop/0.5,0,1))
    spec_s=float(np.clip(1-spec_cost_norm(m['params']),0,1))
    return lat_s,comp_s,reg_s,spec_s
def fis_eval(acc,f1): return tsk_2in(acc,f1,C_ACC,S_ACC,C_F1,S_F1,TSK_EVAL)
def fis_ood(auroc,fpr_inv): return tsk_2in(auroc,fpr_inv,C_AU,S_AU,C_FP,S_FP,TSK_OOD)
def fis_perf(l,c,r,s): return tsk_4in(l,c,r,s,C_STD,S_STD,C_STD,S_STD,C_STD,S_STD,C_STD,S_STD,TSK_PERF)
def fis_sel_from_m(m):
    fe=fis_eval(m['accuracy'],m['f1'])
    fo=fis_ood(m['auroc'],1-m['fpr'])
    l,c,r,s=perf_inputs(m)
    fp=fis_perf(l,c,r,s)
    q=tsk_3in(fe,fo,fp,C_STD,S_STD,C_STD,S_STD,C_STD,S_STD,TSK_SEL)
    return q,fe,fo,fp
print('TSK OK')

TSK OK


In [8]:
# [7] Classes de problema MOO — Fuzzy (4 obj) e Puro (3 obj)
# A mesma classe é reutilizada por todos os algoritmos;
# o que muda entre as rodadas é apenas o objeto `algorithm`.

class ESC50ProblemFuzzy(ElementwiseProblem):
    """4 objetivos: [1-acc, custo, 1-auroc, 1-fis_quality]."""
    def __init__(self):
        super().__init__(n_var=N_VAR,n_obj=4,n_constr=0,xl=xl,xu=xu)
    def _evaluate(self,x,out,*a,**k):
        m=fast_eval(x); q,_,_,_=fis_sel_from_m(m)
        out['F']=np.array([1.0-m['accuracy'],compute_cost(m),1.0-m['auroc'],1.0-q])

class ESC50ProblemPuro(ElementwiseProblem):
    """3 objetivos brutos: [1-acc, custo, 1-auroc]."""
    def __init__(self):
        super().__init__(n_var=N_VAR,n_obj=3,n_constr=0,xl=xl,xu=xu)
    def _evaluate(self,x,out,*a,**k):
        m=fast_eval(x)
        out['F']=np.array([1.0-m['accuracy'],compute_cost(m),1.0-m['auroc']])

print('Problemas MOO OK')

Problemas MOO OK


In [9]:
# [8] Fábrica de algoritmos MOO
# Retorna um objeto pymoo Algorithm dado o tag e o número de objetivos.
# Todos usam os mesmos operadores de crossover/mutação para comparação justa.

def make_algorithm(tag, n_obj):
    """
    tag    : 'nsga2' | 'nsga3' | 'moead' | 'sms' | 'rnsga2'
    n_obj  : 3 (puro) ou 4 (fuzzy)
    Retorna: instância do algoritmo pymoo pronta para minimize().
    """
    xover = SBX(prob=0.9, eta=SBX_ETA)
    mut   = PM(eta=PM_ETA)
    samp  = FloatRandomSampling()

    if tag == 'nsga2':
        return NSGA2(
            pop_size=POP, sampling=samp, crossover=xover, mutation=mut,
            eliminate_duplicates=True
        )

    elif tag == 'nsga3':
        # Pontos de referência estruturados no simplex unitário
        # n_partitions ajustado para que o número de pontos fique ~POP
        n_part = {3: 5, 4: 4}.get(n_obj, 4)  # 3obj→21pts, 4obj→35pts
        ref_dirs = get_reference_directions('das-dennis', n_obj, n_partitions=n_part)
        return NSGA3(
            pop_size=max(POP, len(ref_dirs)),
            ref_dirs=ref_dirs,
            sampling=samp, crossover=xover, mutation=mut,
            eliminate_duplicates=True
        )

    elif tag == 'moead':
        # MOEA/D com decomposição de Tchebycheff
        # n_neighbors: vizinhança local para atualização de sub-problemas
        n_part = {3: 5, 4: 4}.get(n_obj, 4)
        ref_dirs = get_reference_directions('das-dennis', n_obj, n_partitions=n_part)
        return MOEAD(
            ref_dirs=ref_dirs,
            n_neighbors=max(5, len(ref_dirs)//5),
            prob_neighbor_mating=0.7,
            sampling=samp, crossover=xover, mutation=mut
        )

    elif tag == 'sms':
        # SMS-EMOA: remove a cada geração o indivíduo com menor
        # contribuição ao hipervolume → excelente cobertura da fronteira.
        return SMSEMOA(
            pop_size=POP, sampling=samp, crossover=xover, mutation=mut,
            eliminate_duplicates=True
        )

    elif tag == 'rnsga2':
        # R-NSGA-II: guiado por pontos de referência de interesse.
        # Definimos 3 pontos que cobrem alta acc, baixo custo e alto auroc.
        ref_points = np.array([
            [0.05, 0.10, 0.05],   # alta acc, baixo custo, alto auroc
            [0.20, 0.05, 0.10],   # acc moderada, custo mínimo
            [0.10, 0.15, 0.03],   # auroc máximo
        ])[:, :n_obj]  # ajusta dimensão para puro(3) ou fuzzy(4)
        if n_obj == 4:
            # Adicionar coluna para 1-fis_quality: alvo = baixo (0.05 → fq=0.95)
            extra = np.array([[0.05],[0.10],[0.05]])
            ref_points = np.hstack([ref_points, extra])
        return RNSGA2(
            pop_size=POP, ref_points=ref_points, epsilon=0.01,
            sampling=samp, crossover=xover, mutation=mut,
            eliminate_duplicates=True
        )

    else:
        raise ValueError(f'Algoritmo desconhecido: {tag}')

# Teste rápido
for tag, label in ALGORITHMS:
    alg3 = make_algorithm(tag, 3)
    alg4 = make_algorithm(tag, 4)
    print(f'  {label:<12} puro={type(alg3).__name__}  fuzzy={type(alg4).__name__}')
print('Fábrica de algoritmos OK')

  NSGA-II      puro=NSGA2  fuzzy=NSGA2
  NSGA-III     puro=NSGA3  fuzzy=NSGA3
  MOEA/D       puro=MOEAD  fuzzy=MOEAD
Fábrica de algoritmos OK


In [10]:
# [9] Callback genérico — funciona para qualquer algoritmo e qualquer n_obj

def extract_metrics(x_vec, origin_tag):
    key=xkey(x_vec); m=EVAL_CACHE.get(key,{})
    if not m: return None
    q,fe,fo,fp=fis_sel_from_m(m); p=m['params']
    return {
        'key':key,'origin':origin_tag,
        'backbone':m['backbone'],'accuracy':m['accuracy'],'f1':m['f1'],
        'auroc':m['auroc'],'fpr':m['fpr'],'cost':compute_cost(m),
        'fis_quality':q,'fis_eval':fe,'fis_ood':fo,'fis_perf':fp,
        'lr':p['lr'],'optimizer':p['optimizer'],'epochs':p['epochs'],
        'wd':p['wd'],'batch_size':p['batch_size'],'dropout':p['dropout'],
        'n_mels':p['n_mels'],'n_fft':p['n_fft'],
        'hop_length':p['hop_length'],'fmax':p['fmax'],
        'obj_err':1.0-m['accuracy'],'obj_cost':compute_cost(m),'obj_fpr':1.0-m['auroc'],
    }

def make_hist():
    return {'gen':[],'nds':[],'hv_gen':[],'hv_archive':[],
            'best_acc':[],'best_f1':[],'best_auroc':[],'best_cost':[],
            'pop_acc_mean':[],'pop_acc_std':[],'pop_cost_mean':[],'pop_auroc_mean':[],
            'fe_best':[],'fo_best':[],'fp_best':[],'fq_best':[],
            'fe_mean':[],'fo_mean':[],'fp_mean':[],'fq_mean':[],
            'all_acc':[],'all_f1':[],'all_auroc':[],'all_cost':[],
            'all_ep':[],'all_bk':[],'all_gen':[],'all_fq':[]}

def make_callback(hist, archive_F, archive_X, n_obj, label):
    hv_ref = HV_REF4 if n_obj == 4 else HV_REF3
    class _Cb(Callback):
        def __init__(self): super().__init__(); self.t0=time.time()
        def notify(self, algorithm):
            Fn=algorithm.opt.get('F')
            for fr,xr in zip(Fn,algorithm.opt.get('X')):
                archive_F.append(fr.copy()); archive_X.append(xr.copy())
            A=np.array(archive_F); mask=pareto_filter(A)
            try:    hv_arc=float(HV(ref_point=hv_ref).do(A[mask]))
            except: hv_arc=0.0
            gmin=Fn.min(0); gmax=Fn.max(0); rng=np.where(gmax-gmin<1e-12,1e-12,gmax-gmin)
            try:    hv_gen=float(HV(ref_point=np.ones(n_obj)*1.1).do((Fn-gmin)/rng))
            except: hv_gen=0.0
            pop_X=algorithm.pop.get('X')
            pac,pf1,pau,pco,pfe,pfo,pfp,pfq=[],[],[],[],[],[],[],[]
            for xp in pop_X:
                k=xkey(xp); m=EVAL_CACHE.get(k,{})
                if not m: continue
                q,fe,fo,fp=fis_sel_from_m(m)
                pac.append(m['accuracy']); pf1.append(m['f1'])
                pau.append(m['auroc']); pco.append(compute_cost(m))
                pfe.append(fe); pfo.append(fo); pfp.append(fp); pfq.append(q)
                pm=m['params']
                hist['all_acc'].append(m['accuracy']); hist['all_f1'].append(m['f1'])
                hist['all_auroc'].append(m['auroc']); hist['all_cost'].append(compute_cost(m))
                hist['all_ep'].append(pm['epochs']); hist['all_bk'].append(m['backbone'])
                hist['all_gen'].append(algorithm.n_gen); hist['all_fq'].append(q)
            hist['gen'].append(algorithm.n_gen); hist['nds'].append(len(Fn))
            hist['hv_gen'].append(hv_gen); hist['hv_archive'].append(hv_arc)
            if pac:
                hist['best_acc'].append(max(pac)); hist['best_f1'].append(max(pf1))
                hist['best_auroc'].append(max(pau)); hist['best_cost'].append(min(pco))
                hist['pop_acc_mean'].append(np.mean(pac)); hist['pop_acc_std'].append(np.std(pac))
                hist['pop_cost_mean'].append(np.mean(pco)); hist['pop_auroc_mean'].append(np.mean(pau))
                hist['fe_best'].append(max(pfe)); hist['fo_best'].append(max(pfo))
                hist['fp_best'].append(max(pfp)); hist['fq_best'].append(max(pfq))
                hist['fe_mean'].append(np.mean(pfe)); hist['fo_mean'].append(np.mean(pfo))
                hist['fp_mean'].append(np.mean(pfp)); hist['fq_mean'].append(np.mean(pfq))
            else:
                for k2 in ['best_acc','best_f1','best_auroc','best_cost','pop_acc_mean',
                           'pop_acc_std','pop_cost_mean','pop_auroc_mean',
                           'fe_best','fo_best','fp_best','fq_best','fe_mean','fo_mean','fp_mean','fq_mean']:
                    hist[k2].append(hist[k2][-1] if hist[k2] else 0)
            t=(time.time()-self.t0)/60
            eta=t/algorithm.n_gen*(GEN-algorithm.n_gen) if algorithm.n_gen>0 else 0
            acc_now=1-Fn[:,0].min() if len(Fn) else 0
            fq_now=max(pfq) if pfq else 0
            print(f"  [{label:<18}] Gen {algorithm.n_gen:2d}/{GEN}  "
                  f"NDS={len(Fn):2d}  acc={acc_now:.4f}  "
                  f"fq={fq_now:.3f}  HV={hv_arc:.4f}  [{t:.1f}min ETA:{eta:.0f}min]",flush=True)
    return _Cb()

print('Callback genérico OK')

Callback genérico OK


In [ ]:
# [10] LOOP PRINCIPAL — executa as 6 rodadas
#
# Estrutura de resultados:
#   RESULTS[tag][variant] = {
#       'archive_F', 'archive_X',   # todas as soluções encontradas
#       'pF', 'pX',                 # NDS do arquivo
#       'hv_final',                 # HV final
#       'hist',                     # histórico por geração
#       'cache_snapshot',           # snapshot do EVAL_CACHE após a rodada
#   }

RESULTS = {tag: {} for tag, _ in ALGORITHMS}

for tag, label in ALGORITHMS:
    for variant in VARIANTS:
        is_fuzzy = (variant == 'fuzzy')
        n_obj    = 4 if is_fuzzy else 3
        hv_ref   = HV_REF4 if is_fuzzy else HV_REF3
        prob     = ESC50ProblemFuzzy() if is_fuzzy else ESC50ProblemPuro()
        run_lbl  = f'{label} ({"Fuzzy" if is_fuzzy else "Puro"})'

        print('\n' + '='*70)
        print(f'  RODADA: {run_lbl}')
        print(f'  Objetivos: {n_obj}  |  pop={POP}  gen={GEN}')
        print('='*70 + '\n')

        archive_F = []; archive_X = []
        hist      = make_hist()

        alg = make_algorithm(tag, n_obj)

        res = minimize(
            prob, alg, get_termination('n_gen', GEN),
            callback=make_callback(hist, archive_F, archive_X, n_obj, run_lbl),
            seed=SEED, save_history=False, verbose=False
        )

        A    = np.array(archive_F)
        mask = pareto_filter(A)
        pF   = A[mask]
        pX   = np.array(archive_X)[mask]
        try:    hv_fin = float(HV(ref_point=hv_ref).do(pF))
        except: hv_fin = 0.0

        RESULTS[tag][variant] = {
            'archive_F': archive_F, 'archive_X': archive_X,
            'pF': pF, 'pX': pX,
            'hv_final': hv_fin,
            'hist': hist,
            'cache_snapshot': dict(EVAL_CACHE),
            'n_obj': n_obj,
        }

        print(f'\n  ✓ {run_lbl}: NDS={mask.sum()}  HV={hv_fin:.6f}  Cache total={len(EVAL_CACHE)}')

print(f'\n\n=== TODAS AS RODADAS CONCLUÍDAS ===')
print(f'Avaliações únicas no cache: {len(EVAL_CACHE)}')


  RODADA: NSGA-II (Fuzzy)
  Objetivos: 4  |  pop=30  gen=15

  TRAIN ResNet18           lr=2.10e-02 ep=72 el/512fft] split=1200/400/400
        acc=0.6350 f1=0.6158 auroc=0.9713
  TRAIN ResNet18           lr=6.03e-02 ep=69 el/1024fft] split=1200/400/400


In [ ]:
# [11] Construção dos DataFrames por rodada e fronteira unificada global

all_rows = []  # todas as soluções NDS de todas as rodadas
df_per_run = {}  # {(tag, variant): DataFrame}

for tag, label in ALGORITHMS:
    for variant in VARIANTS:
        r = RESULTS[tag][variant]
        origin_tag = f'{label} ({"Fuzzy" if variant=="fuzzy" else "Puro"})'
        rows = []
        seen = set()
        for x in r['pX']:
            k = xkey(x)
            if k in seen: continue
            seen.add(k)
            row = extract_metrics(x, origin_tag)
            if row:
                row['algorithm'] = label
                row['variant']   = variant
                rows.append(row)
        df = pd.DataFrame(rows)
        df_per_run[(tag, variant)] = df
        all_rows.extend(rows)

df_all = pd.DataFrame(all_rows)

# ── Fronteira Pareto UNIFICADA (espaço original 3 objetivos) ──
# Remove duplicatas por chave antes de calcular dominância
df_unique = df_all.drop_duplicates(subset='key', keep='first').copy().reset_index(drop=True)
F_all = df_unique[['obj_err','obj_cost','obj_fpr']].values
nds_mask = pareto_filter(F_all)
df_unified   = df_unique[nds_mask].copy().reset_index(drop=True)
df_dominated = df_unique[~nds_mask].copy().reset_index(drop=True)

print(f'Soluções únicas avaliadas: {len(df_unique)}')
print(f'Fronteira Pareto Unificada: {len(df_unified)} NDS')
print(f'Dominadas: {len(df_dominated)}')
print()
print('NDS por rodada (origin):')
print(df_unified['origin'].value_counts().to_string())
print()
print('NDS por algoritmo (independente de variante):')
print(df_unified['algorithm'].value_counts().to_string())
print()
print('NDS por variante (fuzzy vs puro):')
print(df_unified['variant'].value_counts().to_string())

# ── Pertencimento à fronteira unificada por método/variante ──
# Uma mesma solução pode ter sido encontrada por mais de uma rodada. Para medir
# contribuição, preservamos esse pertencimento em vez de atribuí-la apenas à
# primeira ocorrência após a deduplicação global.
unified_keys = set(df_unified['key'])
rows_unified_by_method = []
for (tag, variant), df_run in df_per_run.items():
    _, algorithm_label = next(item for item in ALGORITHMS if item[0] == tag)
    subset = df_run[df_run['key'].isin(unified_keys)].copy()
    subset['algorithm'] = algorithm_label
    subset['variant'] = variant
    rows_unified_by_method.append(subset)

df_unified_by_method = pd.concat(rows_unified_by_method, ignore_index=True)
print(f'Registros de pertencimento ao Pareto por método: {len(df_unified_by_method)}')

In [ ]:
# [12] Tabela comparativa de HV final e NDS por rodada

rows_hv = []
for tag, label in ALGORITHMS:
    for variant in VARIANTS:
        r = RESULTS[tag][variant]
        df_run = df_per_run[(tag, variant)]
        # Quantas soluções desta rodada ficaram na fronteira unificada
        keys_run = set(df_run['key'].values)
        n_in_unified = df_unified['key'].isin(keys_run).sum()
        rows_hv.append({
            'Algoritmo': label,
            'Variante': 'Fuzzy' if variant=='fuzzy' else 'Puro',
            'N_obj': r['n_obj'],
            'HV_final': round(r['hv_final'], 6),
            'NDS_proprio': len(df_run),
            'NDS_na_unificada': int(n_in_unified),
            'Pct_unificada': round(100*n_in_unified/max(len(df_unified),1), 1),
            'Acc_media': round(df_run['accuracy'].mean()*100, 2) if len(df_run) else 0,
            'F1_media':  round(df_run['f1'].mean()*100, 2)       if len(df_run) else 0,
            'AUROC_media': round(df_run['auroc'].mean(), 4)       if len(df_run) else 0,
            'Custo_medio': round(df_run['cost'].mean(), 4)        if len(df_run) else 0,
            'FQ_media':  round(df_run['fis_quality'].mean(), 4)  if len(df_run) else 0,
        })

df_summary = pd.DataFrame(rows_hv).sort_values(['NDS_na_unificada','HV_final'], ascending=[False,False])
df_summary.to_csv(str(CSV_DIR/'resumo_algoritmos.csv'), index=False)

print('=== TABELA COMPARATIVA ===')
print(df_summary.to_string(index=False))

In [ ]:
# [13] Salvar todos os CSVs
# Tabela única para caracterizar a fronteira global.
df_unique['is_nds'] = nds_mask
df_unique.to_csv(CSV_DIR / 'todas_solucoes_unicas.csv', index=False)
df_unified.to_csv(CSV_DIR / 'pareto_unificado.csv', index=False)

# Tabelas com pertencimento por método/variante, usadas nas comparações.
# `df_all` contém as NDS próprias de cada rodada; portanto, a taxa calculada
# posteriormente significa: fração das NDS da rodada que permaneceu na
# fronteira unificada.
df_all.to_csv(CSV_DIR / 'solucoes_nds_por_metodo.csv', index=False)
df_unified_by_method.to_csv(CSV_DIR / 'pareto_unificado_por_metodo.csv', index=False)

for (tag, variant), df in df_per_run.items():
    fname = f'nds_{tag}_{variant}.csv'
    df.to_csv(CSV_DIR / fname, index=False)

print(f'CSVs salvos em {CSV_DIR}/')
for file in sorted(CSV_DIR.iterdir()):
    print(f'  {file.name}')

## Métricas e visualizações finais

As células seguintes foram integradas a partir de `visualizations.ipynb`. Para evitar ambiguidade, **N_avaliadas** representa o número de soluções não dominadas produzidas pela própria rodada antes da unificação; a taxa de chegada ao Pareto indica quantas dessas soluções permaneceram na fronteira global.

In [ ]:
# ============================================================
# Carregamento dos dados e organização metodológica
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = CSV_DIR
FIG_DIR = PLOTS_DIR / "figuras_dissertacao"
FIG_DIR.mkdir(exist_ok=True)

# Arquivos principais
todas = pd.read_csv(DATA_DIR / "solucoes_nds_por_metodo.csv")
pareto = pd.read_csv(DATA_DIR / "pareto_unificado_por_metodo.csv")
resumo = pd.read_csv(DATA_DIR / "resumo_algoritmos.csv")

# Padronização dos nomes
for df in [todas, pareto]:
    df["algorithm"] = df["algorithm"].astype(str)
    df["variant"] = df["variant"].astype(str).str.lower()

# Coluna mais legível
for df in [todas, pareto]:
    df["Metodo"] = df["algorithm"] + " + " + df["variant"].map({
        "fuzzy": "Agregador Fuzzy",
        "puro": "Puro"
    })

# ============================================================
# Observação metodológica importante:
# Métricas FIS / fuzzy só devem ser interpretadas na variante fuzzy.
# Para a variante pura, elas não representam uma avaliação fuzzy real.
# ============================================================

fis_cols = ["fis_quality", "fis_eval", "fis_ood", "fis_perf"]

for col in fis_cols:
    if col in todas.columns:
        todas.loc[todas["variant"] != "fuzzy", col] = np.nan
    if col in pareto.columns:
        pareto.loc[pareto["variant"] != "fuzzy", col] = np.nan

print("Total de NDS próprias das rodadas:", len(todas))
print("Total de registros no Pareto unificado por método:", len(pareto))

display(todas.head())
display(pareto.head())

In [ ]:
# ============================================================
# Tabela: proporção de cada variante na fronteira de Pareto
# ============================================================

# Total de soluções NDS próprias por algoritmo e variante
avaliadas = (
    todas
    .groupby(["algorithm", "variant"])
    .size()
    .reset_index(name="N_avaliadas")
)

# Total no Pareto unificado por algoritmo e variante
no_pareto = (
    pareto
    .groupby(["algorithm", "variant"])
    .size()
    .reset_index(name="N_pareto")
)

# Combina as informações
comparacao = avaliadas.merge(
    no_pareto,
    on=["algorithm", "variant"],
    how="left"
)

comparacao["N_pareto"] = comparacao["N_pareto"].fillna(0).astype(int)

# Taxa de chegada ao Pareto: entre as NDS próprias daquele método,
# quantas chegaram ao Pareto unificado?
comparacao["Taxa_pareto"] = comparacao["N_pareto"] / comparacao["N_avaliadas"]

# Proporção dentro de cada algoritmo:
# entre as soluções de Pareto daquele algoritmo, qual parcela veio da variante fuzzy/pura?
comparacao["Total_pareto_algoritmo"] = (
    comparacao
    .groupby("algorithm")["N_pareto"]
    .transform("sum")
)

comparacao["Proporcao_no_pareto_do_algoritmo"] = np.where(
    comparacao["Total_pareto_algoritmo"] > 0,
    comparacao["N_pareto"] / comparacao["Total_pareto_algoritmo"],
    0
)

comparacao["Variante"] = comparacao["variant"].map({
    "fuzzy": "Agregador Fuzzy",
    "puro": "Puro"
})

comparacao = comparacao.sort_values(["algorithm", "variant"])

display(comparacao)

In [ ]:
# ============================================================
# Figura principal:
# Proporção da fronteira de Pareto por variante em cada algoritmo
# ============================================================

pivot_prop = (
    comparacao
    .pivot(index="algorithm", columns="Variante", values="Proporcao_no_pareto_do_algoritmo")
    .fillna(0)
)

# Garante ordem visual
ordem_colunas = ["Puro", "Agregador Fuzzy"]
pivot_prop = pivot_prop[[c for c in ordem_colunas if c in pivot_prop.columns]]

ax = pivot_prop.plot(
    kind="bar",
    stacked=True,
    figsize=(8.5, 5.5),
    edgecolor="black",
    linewidth=0.6
)

plt.title(
    "Composição da Fronteira de Pareto por Algoritmo e Estratégia de Agregação",
    fontsize=13,
    pad=12
)

plt.xlabel("Algoritmo de otimização", fontsize=11)
plt.ylabel("Proporção na fronteira de Pareto", fontsize=11)

plt.ylim(0, 1.05)
plt.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.45)
plt.legend(title="Estratégia", fontsize=10, title_fontsize=10)

# Rótulos percentuais
for container in ax.containers:
    labels = []
    for value in container.datavalues:
        labels.append(f"{value*100:.1f}%" if value > 0 else "")
    ax.bar_label(container, labels=labels, label_type="center", fontsize=9)

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(FIG_DIR / "proporcao_pareto_por_algoritmo_variante.png", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "proporcao_pareto_por_algoritmo_variante.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# Número absoluto de soluções no Pareto unificado
# ============================================================

pivot_count = (
    comparacao
    .pivot(index="algorithm", columns="Variante", values="N_pareto")
    .fillna(0)
)

pivot_count = pivot_count[[c for c in ordem_colunas if c in pivot_count.columns]]

ax = pivot_count.plot(
    kind="bar",
    figsize=(8.5, 5.5),
    edgecolor="black",
    linewidth=0.6
)

plt.title(
    "Número de Soluções na Fronteira de Pareto Unificada",
    fontsize=13,
    pad=12
)

plt.xlabel("Algoritmo de otimização", fontsize=11)
plt.ylabel("Número de soluções no Pareto unificado", fontsize=11)

plt.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.45)
plt.legend(title="Estratégia", fontsize=10, title_fontsize=10)

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", fontsize=9)

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(FIG_DIR / "numero_solucoes_pareto_por_algoritmo_variante.png", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "numero_solucoes_pareto_por_algoritmo_variante.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# Taxa de chegada ao Pareto
# ============================================================

pivot_taxa = (
    comparacao
    .pivot(index="algorithm", columns="Variante", values="Taxa_pareto")
    .fillna(0)
)

pivot_taxa = pivot_taxa[[c for c in ordem_colunas if c in pivot_taxa.columns]]

ax = pivot_taxa.plot(
    kind="bar",
    figsize=(8.5, 5.5),
    edgecolor="black",
    linewidth=0.6
)

plt.title(
    "Eficiência das Estratégias em Gerar Soluções Não Dominadas",
    fontsize=13,
    pad=12
)

plt.xlabel("Algoritmo de otimização", fontsize=11)
plt.ylabel("Taxa de soluções no Pareto unificado", fontsize=11)

plt.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.45)
plt.legend(title="Estratégia", fontsize=10, title_fontsize=10)

for container in ax.containers:
    labels = [f"{v*100:.1f}%" if v > 0 else "" for v in container.datavalues]
    ax.bar_label(container, labels=labels, fontsize=9)

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(FIG_DIR / "taxa_chegada_pareto_por_algoritmo_variante.png", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "taxa_chegada_pareto_por_algoritmo_variante.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# Ganho relativo do agregador fuzzy em relação à abordagem pura
# ============================================================

ganho = (
    comparacao
    .pivot(index="algorithm", columns="variant", values="N_pareto")
    .fillna(0)
    .reset_index()
)

if "fuzzy" not in ganho.columns:
    ganho["fuzzy"] = 0

if "puro" not in ganho.columns:
    ganho["puro"] = 0

ganho["Diferenca_fuzzy_menos_puro"] = ganho["fuzzy"] - ganho["puro"]

ganho["Interpretacao"] = np.where(
    ganho["Diferenca_fuzzy_menos_puro"] > 0,
    "Fuzzy superior",
    np.where(
        ganho["Diferenca_fuzzy_menos_puro"] < 0,
        "Puro superior",
        "Empate"
    )
)

display(ganho)

plt.figure(figsize=(8, 5.2), dpi=150)

plt.bar(
    ganho["algorithm"],
    ganho["Diferenca_fuzzy_menos_puro"],
    edgecolor="black",
    linewidth=0.6
)

plt.axhline(0, color="black", linewidth=1.0)

plt.title(
    "Diferença de Contribuição no Pareto: Fuzzy menos Puro",
    fontsize=13,
    pad=12
)

plt.xlabel("Algoritmo de otimização", fontsize=11)
plt.ylabel("Diferença no número de soluções não dominadas", fontsize=11)

plt.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.45)

for i, row in ganho.iterrows():
    value = row["Diferenca_fuzzy_menos_puro"]
    plt.text(
        i,
        value + (0.15 if value >= 0 else -0.25),
        f"{value:.0f}",
        ha="center",
        va="bottom" if value >= 0 else "top",
        fontsize=10
    )

plt.tight_layout()

plt.savefig(FIG_DIR / "ganho_fuzzy_menos_puro_pareto.png", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "ganho_fuzzy_menos_puro_pareto.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# Matriz de contribuição para o Pareto unificado
# ============================================================

matriz = (
    comparacao
    .pivot(index="algorithm", columns="Variante", values="N_pareto")
    .fillna(0)
)

matriz = matriz[[c for c in ordem_colunas if c in matriz.columns]]

plt.figure(figsize=(7, 4.8), dpi=150)

plt.imshow(matriz.values, aspect="auto")

plt.title("Matriz de Contribuição para o Pareto Unificado", fontsize=13, pad=12)
plt.xlabel("Estratégia", fontsize=11)
plt.ylabel("Algoritmo", fontsize=11)

plt.xticks(range(len(matriz.columns)), matriz.columns, rotation=0)
plt.yticks(range(len(matriz.index)), matriz.index)

for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):
        plt.text(
            j,
            i,
            int(matriz.iloc[i, j]),
            ha="center",
            va="center",
            fontsize=12,
            color="white" if matriz.iloc[i, j] > matriz.values.max() / 2 else "black"
        )

plt.colorbar(label="Número de soluções no Pareto")
plt.tight_layout()

plt.savefig(FIG_DIR / "matriz_contribuicao_pareto.png", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "matriz_contribuicao_pareto.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# Tabela LaTeX: contribuição para o Pareto unificado
# ============================================================

tabela_latex = comparacao.copy()

tabela_latex["Taxa_pareto_%"] = 100 * tabela_latex["Taxa_pareto"]
tabela_latex["Proporcao_no_algoritmo_%"] = 100 * tabela_latex["Proporcao_no_pareto_do_algoritmo"]

tabela_latex = tabela_latex[
    [
        "algorithm",
        "Variante",
        "N_avaliadas",
        "N_pareto",
        "Taxa_pareto_%",
        "Proporcao_no_algoritmo_%"
    ]
].rename(columns={
    "algorithm": "Algoritmo",
    "Variante": "Estratégia",
    "N_avaliadas": "Soluções avaliadas",
    "N_pareto": "Soluções no Pareto",
    "Taxa_pareto_%": "Taxa no Pareto (%)",
    "Proporcao_no_algoritmo_%": "Proporção no algoritmo (%)"
})

display(tabela_latex)

latex_path = FIG_DIR / "tabela_contribuicao_pareto.tex"

with open(latex_path, "w", encoding="utf-8") as f:
    f.write(
        tabela_latex.to_latex(
            index=False,
            float_format="%.2f",
            caption="Contribuição das estratégias pura e fuzzy para a fronteira de Pareto unificada.",
            label="tab:contribuicao_pareto_fuzzy_puro"
        )
    )

print(f"Tabela LaTeX salva em: {latex_path}")

In [ ]:
# [19] Salvar checkpoint geral
summary_dict = {}
for tag, label in ALGORITHMS:
    summary_dict[tag] = {}
    for variant in VARIANTS:
        r = RESULTS[tag][variant]
        df_run = df_per_run[(tag, variant)]
        keys_run = set(df_run['key'].values)
        n_in = int(df_unified['key'].isin(keys_run).sum())
        summary_dict[tag][variant] = {
            'hv_final': r['hv_final'],
            'n_nds_proprio': len(df_run),
            'n_nds_na_unificada': n_in,
            'pct_unificada': round(100*n_in/max(len(df_unified),1),1),
        }

log = {
    'version': 'v23', 'dataset': 'esc50',
    'nsga2': {'pop':POP,'gen':GEN},
    'algoritmos': [label for _,label in ALGORITHMS],
    'n_rodadas': len(ALGORITHMS) * len(VARIANTS),
    'avaliacoes_unicas': len(EVAL_CACHE),
    'n_nds_unificada': len(df_unified),
    'resultados': summary_dict,
    'nds_unificada_por_algoritmo': df_unified_by_method['algorithm'].value_counts().to_dict(),
    'nds_unificada_por_variante': df_unified_by_method['variant'].value_counts().to_dict(),
}
with open(str(LOGS_DIR/'resultado_esc50_v23.json'),'w') as f:
    json.dump(log,f,indent=2,default=str)

torch.save({
    'version':'v23','eval_cache_keys':list(EVAL_CACHE.keys()),
    'summary': summary_dict,
}, str(CKPT_DIR/'comparacao_algoritmos_esc50_v23.pth'))

print(f'Checkpoint salvo em {BASE_DIR}/')

In [ ]:
# [20] ANÁLISE TEXTUAL INTERPRETATIVA — pronta para dissertação/artigo

print('='*80)
print('ANÁLISE COMPARATIVA ESC-50 v23 — 3 Algoritmos MOO × 2 Variantes (Fuzzy / Puro)')
print('Dataset: ESC-50 | Objetivos Fuzzy: 4 | Objetivos Puro: 3')
print('='*80)

# Melhor algoritmo + variante
best_row = df_summary.iloc[0]
worst_row = df_summary.iloc[-1]

# Fuzzy ganhou de puro?
n_fz_total = df_unified_by_method['variant'].value_counts().get('fuzzy', 0)
n_pu_total = df_unified_by_method['variant'].value_counts().get('puro', 0)
pct_fz_total = 100*n_fz_total/max(len(df_unified_by_method),1)
pct_pu_total = 100*n_pu_total/max(len(df_unified_by_method),1)

print(f"""
1. VISÃO GERAL DA FRONTEIRA UNIFICADA
   ─────────────────────────────────────────────────────────────────
   Soluções NDS únicas reunidas       : {len(df_unique)}
   Soluções NDS na fronteira unificada: {len(df_unified)}
   Dominadas:                           {len(df_dominated)}

   Por variante:
     Fuzzy  (4 objetivos): {n_fz_total} NDS  ({pct_fz_total:.1f}%)
     Puro   (3 objetivos): {n_pu_total} NDS  ({pct_pu_total:.1f}%)

   {'[✓] A variante Fuzzy contribuiu com mais soluções NDS, indicando que' if n_fz_total >= n_pu_total else '[~] A variante Puro contribuiu com mais soluções NDS, indicando que'}
   {'o Agregador Fuzzy guiou melhor a busca dos algoritmos.' if n_fz_total >= n_pu_total else 'os objetivos brutos foram suficientes para encontrar boas soluções.'}

2. MELHOR E PIOR RODADA (por NDS na fronteira unificada)
   ─────────────────────────────────────────────────────────────────
   Melhor : {best_row['Algoritmo']:12s} ({best_row['Variante']:5s})  """
      f"NDS_unif={best_row['NDS_na_unificada']}  HV={best_row['HV_final']:.4f}  "
      f"Acc={best_row['Acc_media']:.1f}%  FQ={best_row['FQ_media']:.3f}")
print(f"   Pior  : {worst_row['Algoritmo']:12s} ({worst_row['Variante']:5s})  "
      f"NDS_unif={worst_row['NDS_na_unificada']}  HV={worst_row['HV_final']:.4f}  "
      f"Acc={worst_row['Acc_media']:.1f}%  FQ={worst_row['FQ_media']:.3f}")

print(f"""
3. ANÁLISE POR ALGORITMO
   ─────────────────────────────────────────────────────────────────""")
for tag, label in ALGORITHMS:
    fz = summary_dict[tag]['fuzzy']
    pu = summary_dict[tag]['puro']
    ganho = fz['n_nds_na_unificada'] - pu['n_nds_na_unificada']
    sinal = '[+Fuzzy]' if ganho > 0 else ('[=Empate]' if ganho==0 else '[-Fuzzy]')
    print(f"   {label:<12}  Fuzzy_NDS={fz['n_nds_na_unificada']:2d} ({fz['pct_unificada']:4.1f}%)"
          f"  Puro_NDS={pu['n_nds_na_unificada']:2d} ({pu['pct_unificada']:4.1f}%)"
          f"  Δ={ganho:+d}  {sinal}")

print(f"""
4. DIFERENÇAS ENTRE OS ALGORITMOS MOO
   ─────────────────────────────────────────────────────────────────
   - NSGA-II   : baseline de crowding distance. Equilibrado, mas pode
     perder diversidade em fronteiras irregulares.
   - NSGA-III  : pontos de referência no simplex; melhor para ≥3 obj.
     Tende a distribuir soluções mais uniformemente na fronteira.
   - MOEA/D    : decompõe em sub-problemas escalares. Ótimo para fronteiras
     convexas; pode ter dificuldade com fronteiras descontínuas.

5. RESPOSTA PRINCIPAL
   ─────────────────────────────────────────────────────────────────
   "Qual algoritmo + variante produziu as soluções mais competitivas"
   "(maior participação na fronteira de Pareto unificada)?"

   RESPOSTA: {best_row['Algoritmo']} + {best_row['Variante']} com {best_row['NDS_na_unificada']} soluções
   na fronteira unificada ({best_row['Pct_unificada']:.1f}% do total NDS).

   A comparação por CONJUNTO de soluções (e não pela melhor solução isolada)
   é a evidência metodológica mais robusta para sustentar a superioridade
   de um algoritmo ou abordagem no contexto da otimização multiobjetivo.
""")
print('='*80)

In [ ]:
# [21] Sumário de arquivos
print(f'\n=== ARQUIVOS GERADOS EM {BASE_DIR} ===')
for sub in [CKPT_DIR, PLOTS_DIR, LOGS_DIR, CSV_DIR]:
    files = list(sub.iterdir())
    print(f'  {sub.name}/ ({len(files)} arquivos)')
    for f in sorted(files):
        print(f'    {f.name}  ({f.stat().st_size//1024}KB)')